# 03  -   CNN: Clasificación de Mosquito


Notebook explicativo que implementa y compara dos enfoques de Deep Learning para la clasificación de imágenes de mosquitos esto por medio de dos modelos:

- Red Neuronal Convolucional (CNN)
- Modelo  basado en Transfer Learning utilizando MobileNetV2

Con el objetivo de determinar que arquitectura detecta mejor las clases:

- Aedes aegypti
- Aedes albopictus
- Anopheles
- Culex

La selección del modelo se realizó por medio del accuracy y F1 Macro

#### Recordatorio
El Transfer Learning es una técnica donde se utiliza un modelo ya entrenado para resolver un problema general, usando este de base para resolver un problema en específico

## 1. Carga de Librerías


In [39]:
import tensorflow as tf
import pandas as pd
import numpy as np

from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras import mixed_precision
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (Input, Rescaling, 
                                     Conv2D, MaxPooling2D, 
                                     Dense, Dropout, 
                                     BatchNormalization, GlobalAveragePooling2D)
from tensorflow.keras.utils import image_dataset_from_directory
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.applications import MobileNetV2
from pathlib import Path
from sklearn.metrics import classification_report, f1_score


## 1.2 Carga de Datos a Utilizar

In [19]:
#Para activar precisión mixta
mixed_precision.set_global_policy("mixed_float16")

#Aquí se define el tamaño de la imagen
IMG_SIZE = (224, 224)

#Tamaño del lote
BATCH_SIZE = 16

#Cantidad de épocas esperadas
EPOCHS = 30

#Se define la semilla.
SEED = 42

print(
    tf.config.list_physical_devices("GPU")
)

#Lee las carpetas necesarias (entrenamiento, validacion y test) y deduce la clase por el nombre de la carpeta.
train_ds = image_dataset_from_directory(
    "../data/processed/entrenamiento",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    seed=SEED
)

val_ds = image_dataset_from_directory(
    "../data/processed/validacion",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    seed=SEED
)

test_ds = image_dataset_from_directory(
    "../data/processed/test",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False #Se usa para que el orden coincida con las etiquetas al evaluar.
)

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Found 5718 files belonging to 4 classes.
Found 1141 files belonging to 4 classes.
Found 1153 files belonging to 4 classes.


Las imágenes se dividen en tres categorías
- Entrenamiento
- Validación
- Prueba

Siendo la primera la que mayor cantidad posee con 5718, además las imágenes se redimensionan a 224 x 224 pixeles.

## 1.3 Guardado de Clases

In [20]:
#Se crea "train_ds"
train_ds = image_dataset_from_directory(
    "../data/processed/entrenamiento",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    seed=SEED
)

class_names = train_ds.class_names #Guarda las 4 clases en orden alfabético

print(class_names) #Imprime el nombre de las 4 clases.

Found 5718 files belonging to 4 classes.
['aegypti', 'albopictus', 'anopheles', 'culex']


Se corrobora que se hayan guardado las cuatro clases de manera correcta

## 1.4 Optimización de Pesos

In [22]:
#Recorre "train_ds" para juntar todas las etiquetas.
labels = []

for _, y in train_ds:

    labels.extend(
        y.numpy()
    )

labels = np.array(labels)

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(labels),
    y=labels
)

class_weights = dict(
    enumerate(class_weights)
)

print(class_weights)

{0: np.float64(0.6062340966921119), 1: np.float64(0.836943793911007), 2: np.float64(3.7033678756476682), 3: np.float64(1.1291469194312795)}


Este aparatado se realiza con el fin de que ninguna clase influya por sobre el resto, ya que los modelos suelen tener mayor prioridad por las clases que poseen menos imágenes que el resto lo cual hace que no aprenda de manera correcta.

En este caso observamos que una posee Anopheles posee un peso mucho mayor que el resto esto hubiera desembocado en que el modelo tenga un sezgo hacia dicha clase y no aprenda correctamente

## 1.5 Optimización de Pipeline

En este apartado en particular se hace una mejora en el rendimiento de los datos durante el entrenamiento

In [23]:
AUTOTUNE = tf.data.AUTOTUNE #Prepara el siguiente lote mientras se entrena con el actual, y TensorFlow elige el nivel de paralelismo automáticamente.

train_ds = train_ds.prefetch(AUTOTUNE)

val_ds = val_ds.prefetch(AUTOTUNE)

test_ds = test_ds.prefetch(AUTOTUNE)

Prefetch permite que TensorFlow prepare el siguiente lotes de imágenes mientras la GPU aún termina de procesar el actual.

AUTOTUNE permite saber el nivel óptimo de paralelismo y precarga según los recursos del sistema, gracias a esto se reduce el tiempo de espera

## 1.6 Data Augmentation

In [15]:
#Capas de volteo horizontal
data_augmentation = tf.keras.Sequential([

    tf.keras.layers.RandomFlip(
        "horizontal"
    ),

    tf.keras.layers.RandomRotation( #Rotación (±15%)
        0.15
    ),

    tf.keras.layers.RandomZoom( #Acercamiento (±15%)
        0.15
    )

])

Recordemos que el "data augmentation" es la creación artificial de datos siendo en este caso el voltear las imágenes para un mayor desempeño del modelo, este con el objetivo de que identifique (en este caso) al mosquito sin importar como esté la imagen.

## 2. Creación del Modelo

En este caso vamos a dos modelos, uno siendo el CNN que se vio en clase y el otro siendo un modelo con MobileNetV2

### 2.1.1 Creación de Modelo CNN

In [24]:
model_cnn = Sequential([

    Input(shape=(224,224,3)),

    Rescaling(1./255), #Para normalizar los píxeles y luego se aplica el augmentation.

    data_augmentation,

    # Bloque 1
    Conv2D(
        32,
        (3,3),
        activation="relu"
    ),

    BatchNormalization(), #Para estabilizar el entrenamiento.

    MaxPooling2D((2,2)), #Reduce el tamaño a la mitad.

#Mismo proceso en cada bloque...

    # Bloque 2
    Conv2D(
        64,
        (3,3),
        activation="relu"
    ),

    BatchNormalization(),

    MaxPooling2D((2,2)),

    # Bloque 3
    Conv2D(
        128,
        (3,3),
        activation="relu"
    ),

    BatchNormalization(),

    MaxPooling2D((2,2)),

Conv2D(
    256,
    (3,3),
    activation="relu"
),

BatchNormalization(),

MaxPooling2D((2,2)),

GlobalAveragePooling2D(), #Convierte los mapas de características en un vector.


#Esta parte se encarga de apagar el 50% de las neuronas para evitar el overfitting.
Dense(
    256,
    activation="relu"
),

Dropout(
    0.5
),

#Apartado para que sea estable con la precisión mixta.
Dense(
    NUM_CLASSES,
    activation="softmax",
    dtype="float32"
)

])

Está compuesta por cuatro bloques convolucionales encargados encargados de extraer características visuales de las imágenes.

Consta con técnicas que permiten que no haya sobreajuste como los son:
- Batch Normalization
- Dropout
- Global Average Pooling

### 2.1.2 Compilación

In [25]:
#Se compila con "adam"
model_cnn.compile(

    optimizer="adam",

    loss="sparse_categorical_crossentropy",

    metrics=["accuracy"]

)

model_cnn.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ rescaling_1 (Rescaling)         │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential (Sequential)         │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 222, 222, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 109, 109, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_6 (Conv2D)               │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_6           │ (None, 52, 52, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_6 (MaxPooling2D)  │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_7 (Conv2D)               │ (None, 24, 24, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_7           │ (None, 24, 24, 256)    │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_7 (MaxPooling2D)  │ (None, 12, 12, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 256)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 256)            │        65,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 4)              │         1,028 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 457,156 (1.74 MB)

 Trainable params: 456,196 (1.74 MB)

 Non-trainable params: 960 (3.75 KB)

### 2.1.3 Callbacks

In [26]:
callbacks = [


    EarlyStopping( #Detiene el entrenamiento si "val_loss" no mejora en 5 épocas y restaura los mejores pesos.

        monitor="val_loss",

        patience=5,

        restore_best_weights=True

    ),

    ReduceLROnPlateau( #Reduce la tasa de aprendizaje a la mitad si no mejora en 2 épocas.

        monitor="val_loss",

        factor=0.5,

        patience=2

    )

]

### 2.1.4 Entrenamiento CNN

In [27]:
history_cnn = model_cnn.fit(

    train_ds,

    validation_data=val_ds,

    epochs=EPOCHS,

    callbacks=callbacks

)


Epoch 1/30


/home/sebas/Vector_IA/.venv/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(
E0000 00:00:1793801340.271786     859 util.cc:131] oneDNN supports DT_HALF only on platforms with AVX-512. Falling back to the default Eigen-based implementation if present.
I0000 00:00:1793801341.103009    1197 cuda_dnn.cc:461] Loaded cuDNN version 92700


358/358 ━━━━━━━━━━━━━━━━━━━━ 46s 105ms/step - accuracy: 0.5766 - loss: 0.8718 - val_accuracy: 0.4452 - val_loss: 3.3251 - learning_rate: 0.0010
Epoch 2/30
358/358 ━━━━━━━━━━━━━━━━━━━━ 34s 95ms/step - accuracy: 0.6794 - loss: 0.6810 - val_accuracy: 0.4759 - val_loss: 2.2895 - learning_rate: 0.0010
Epoch 3/30
358/358 ━━━━━━━━━━━━━━━━━━━━ 36s 101ms/step - accuracy: 0.7929 - loss: 0.4792 - val_accuracy: 0.8221 - val_loss: 0.4851 - learning_rate: 0.0010
Epoch 4/30
358/358 ━━━━━━━━━━━━━━━━━━━━ 34s 96ms/step - accuracy: 0.8283 - loss: 0.4172 - val_accuracy: 0.6933 - val_loss: 0.6457 - learning_rate: 0.0010
Epoch 5/30
358/358 ━━━━━━━━━━━━━━━━━━━━ 34s 95ms/step - accuracy: 0.8417 - loss: 0.3933 - val_accuracy: 0.4794 - val_loss: 4.8251 - learning_rate: 0.0010
Epoch 6/30
358/358 ━━━━━━━━━━━━━━━━━━━━ 34s 95ms/step - accuracy: 0.8552 - loss: 0.3423 - val_accuracy: 0.5197 - val_loss: 3.1174 - learning_rate: 5.0000e-04
Epoch 7/30
358/358 ━━━━━━━━━━━━━━━━━━━━ 34s 96ms/step - accuracy: 0.8666 - loss: 

El modelo alcanzó un total de 88% de precisión, a continuación crearemos otro modelo pero con MobileNetV2

### 2.2.1 Creación de Modelo MobileNetV2

Como mencionamos anteriormente MobileNetV2 utiliza la arquitectura de Image Net, esto nos permite utilizar características visuales aprendidas a partir de millones de imágenes por lo cual se obtienen mejores resultados aún con conjuntos de datos pequeños

In [28]:
base_model = MobileNetV2( #Carga el modelo con pesos de ImageNet

    weights="imagenet",

    include_top=False,

    input_shape=(224,224,3)

)

base_model.trainable = False #Congela los pesos

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


### 2.2.2 Modelo Completo

In [29]:
model_mobilenet = Sequential([

    Input(
        shape=(224,224,3)
    ),

    Rescaling(1./255),

    data_augmentation,

    base_model,

    GlobalAveragePooling2D(),

    Dense(
        128,
        activation="relu"
    ),

    Dropout(
        0.5
    ),

    Dense(
        NUM_CLASSES,
        activation="softmax",
        dtype="float32"
    )

])

### 2.2.3 Compilación

In [30]:
model_mobilenet.compile(

    optimizer="adam",

    loss="sparse_categorical_crossentropy",

    metrics=["accuracy"]

)

model_mobilenet.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ rescaling_2 (Rescaling)         │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential (Sequential)         │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_2      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 4)              │           516 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,422,468 (9.24 MB)

 Trainable params: 164,484 (642.52 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

### 2.2.4 Entrenamiento del Modelo

In [31]:
history_mobile = model_mobilenet.fit(

    train_ds,

    validation_data=val_ds,

    epochs=EPOCHS,

    callbacks=callbacks,

    class_weight=class_weights

)

Epoch 1/30
358/358 ━━━━━━━━━━━━━━━━━━━━ 28s 63ms/step - accuracy: 0.7139 - loss: 0.6395 - val_accuracy: 0.8370 - val_loss: 0.3938 - learning_rate: 0.0010
Epoch 2/30
358/358 ━━━━━━━━━━━━━━━━━━━━ 21s 60ms/step - accuracy: 0.8164 - loss: 0.3724 - val_accuracy: 0.8861 - val_loss: 0.2824 - learning_rate: 0.0010
Epoch 3/30
358/358 ━━━━━━━━━━━━━━━━━━━━ 21s 58ms/step - accuracy: 0.8464 - loss: 0.3252 - val_accuracy: 0.9071 - val_loss: 0.2527 - learning_rate: 0.0010
Epoch 4/30
358/358 ━━━━━━━━━━━━━━━━━━━━ 19s 54ms/step - accuracy: 0.8596 - loss: 0.2897 - val_accuracy: 0.8992 - val_loss: 0.2707 - learning_rate: 0.0010
Epoch 5/30
358/358 ━━━━━━━━━━━━━━━━━━━━ 20s 54ms/step - accuracy: 0.8722 - loss: 0.2599 - val_accuracy: 0.9001 - val_loss: 0.2463 - learning_rate: 0.0010
Epoch 6/30
358/358 ━━━━━━━━━━━━━━━━━━━━ 21s 56ms/step - accuracy: 0.8881 - loss: 0.2350 - val_accuracy: 0.9062 - val_loss: 0.2298 - learning_rate: 0.0010
Epoch 7/30
358/358 ━━━━━━━━━━━━━━━━━━━━ 20s 55ms/step - accuracy: 0.8865 - l

Observamos que en la parte del entrenamiento el segundo modelo obtuvo mejores resultados quedando de la siguiente manera:

- CNN: accuracy: 0.8802 - loss: 0.2856 - val_accuracy: 0.7713 - val_loss: 0.6301

- MobileNetV2: accuracy: 0.9575 - loss: 0.0953 - val_accuracy: 0.9395 - val_loss: 0.1622

Ahora vamos a comprobar su desempeño con Test.

## 3 Comparación en Test

### 3.1 CNN

In [32]:
loss_cnn, acc_cnn = model_cnn.evaluate( #Para evaluar la precisión del modelo.
    test_ds
)

print(
    f"CNN Accuracy: {acc_cnn:.4f}"
)

73/73 ━━━━━━━━━━━━━━━━━━━━ 2s 24ms/step - accuracy: 0.8161 - loss: 0.4270
CNN Accuracy: 0.8161


### 3.2 MobileNetV2

In [33]:
loss_mobile, acc_mobile = model_mobilenet.evaluate(
    test_ds
)

print(
    f"MobileNetV2 Accuracy: {acc_mobile:.4f}"
)

73/73 ━━━━━━━━━━━━━━━━━━━━ 3s 43ms/step - accuracy: 0.9315 - loss: 0.1602
MobileNetV2 Accuracy: 0.9315


Observamos que en el test ahora quedaron:

- CNN: 0.8161

- MobileNetV2: Accuracy: 0.9315

## 4. F1 Macro

### 4.1 CNN

In [35]:
y_true = np.concatenate(
    [y for _, y in test_ds]
)

y_pred_cnn = np.argmax( #Aquí toma la clase con mayor probabilidad.

    model_cnn.predict(
        test_ds
    ),

    axis=1

)

print(
    classification_report( #Aquí se muestra la precisión, recall y F1 por clase.
        y_true,
        y_pred_cnn,
        target_names=class_names
    )
)

print(
    "CNN F1 Macro:", #Este promedia el F1 de las clases con el mismo peso.
    f1_score(
        y_true,
        y_pred_cnn,
        average="macro"
    )
)

73/73 ━━━━━━━━━━━━━━━━━━━━ 2s 20ms/step
              precision    recall  f1-score   support

     aegypti       0.81      0.84      0.83       471
  albopictus       0.98      0.84      0.91       352
   anopheles       0.10      0.01      0.02        77
       culex       0.70      0.98      0.82       253

    accuracy                           0.82      1153
   macro avg       0.65      0.67      0.64      1153
weighted avg       0.79      0.82      0.79      1153

CNN F1 Macro: 0.6431906431529198


### 4.2 MobileNetV2

In [36]:
y_pred_mobile = np.argmax(

    model_mobilenet.predict(
        test_ds
    ),

    axis=1

)

print(
    classification_report(
        y_true,
        y_pred_mobile,
        target_names=class_names
    )
)

print(
    "MobileNetV2 F1 Macro:",
    f1_score(
        y_true,
        y_pred_mobile,
        average="macro"
    )
)

73/73 ━━━━━━━━━━━━━━━━━━━━ 4s 46ms/step
              precision    recall  f1-score   support

     aegypti       0.96      0.87      0.91       471
  albopictus       0.87      0.96      0.91       352
   anopheles       0.90      1.00      0.94        77
       culex       0.99      0.98      0.98       253

    accuracy                           0.93      1153
   macro avg       0.93      0.95      0.94      1153
weighted avg       0.93      0.93      0.93      1153

MobileNetV2 F1 Macro: 0.9391826984144646


#### RECORDATORIO

El F1 Score es una métrica que combina la Precisión y el Recall en una sola métrica, esto con el objetivo de mostrar de manera "honesta" el desempeño que tuvo un modelo

El F1 Macro le asigna el mismo valor a las clases que se están trabajando, entonces aunque una clase posea mas elementos que otra todas tienen el mismo valor

El F1 de ámbos modelos fue de:
- CNN F1 Macro: 0.6431906431529198

- MobileNetV2 F1 Macro: 0.9391826984144646

Observamos que el modelo con mejor desempeño fue el MobileNetV2 por lo cual ese será el modelo con el que se va a trabajar, ahora, se va a subir ámbos modelos como un archivo CSV para evidencia concreta de su realización y resultados.
  

## 5. Cargado del Modelo

### 5.1 Cargado de MobileNetV2

In [37]:
model_mobilenet.save(
    "../models/modelo_final_mosquitos.keras"
)

### 5.2 Historial MobileNetV2

In [40]:
pd.DataFrame(
    history_mobile.history
).to_csv(
    "../models/mobilenetv2_history.csv",
    index=False
)

### 5.3 Historial CNN

In [41]:
pd.DataFrame(
    history_cnn.history
).to_csv(
    "../models/cnn_history.csv",
    index=False
)

Con esto tenemos ámbos modelos guardados en archivos CSV en el cual guarda las estadísticas que este obtuvo (accuracy, loss, val_accuracy, val_loss) no solo del final sino que de todos los epochs que realizó

### 5.4 Verificación

In [42]:
for archivo in Path("../models").iterdir():
    print(archivo)

../models/cnn_history.csv
../models/modelo_final_mosquitos.keras
../models/mobilenetv2_history.csv
